# Экспорт данных из DATA_MOS


**DATA_MOS** — это [портал открытых данных Правительства Москвы](https://data.mos.ru/). На нём опубликованы сотни наборов данных о городе: парки, поликлиники, спортивные объекты, памятники и многое другое. Часть из них — пространственные, то есть у объектов есть координаты, и их можно нанести на карту.

Забирать данные с портала можно вручную (кнопкой «Экспорт»), но удобнее — через **API**, обращаясь к сервису прямо из кода.

🔗 **API** (application programming interface) — это «посредник», который позволяет одной программе обращаться к другой. Мы отправляем сервису запрос с нужными параметрами и получаем в ответ данные.

Как и в разделе про геокодирование, процесс сводится к нескольким шагам:

1. Формируем запрос к API (какой набор данных нужен и с каким ключом доступа)
2. Отправляем запрос и получаем ответ
3. Обрабатываем ответ — превращаем его в таблицу и в набор пространственных данных

---

Для работы с API DATA_MOS нужен **API-ключ**: его можно бесплатно получить в личном кабинете после регистрации на [портале](https://data.mos.ru/). Подробности — в [документации к API](https://apidata.mos.ru/Docs).


## 0. Импортируем библиотеки


In [ ]:
# Для работы с табличными и пространственными данными
import pandas as pd
import geopandas as gpd

# Для запросов к API
import requests

## 1. Запрос данных через API


Разберём процесс на примере одного набора данных: определим параметры запроса, отправим его и посмотрим, что пришло в ответе.


### 1.1 Определяем параметры запроса


Чтобы забрать данные, сервису нужно объяснить, **что именно** мы хотим:

- **номер набора данных** (`data_set`) — идентификатор конкретного датасета на портале. Его видно в адресе набора на сайте data.mos.ru (например, `.../opendata/1193/...`).
- **API-ключ** (`datamos_api`) — ключ доступа, полученный в личном кабинете.

Из этих частей мы собираем **URL** — адрес, по которому будем обращаться к API.


In [ ]:
datamos_api = ''   # Ваш API-ключ с портала data.mos.ru
data_set = 1193    # Номер набора данных (пример; замените на нужный)

# Собираем адрес запроса: просим у API все объекты (features) выбранного набора
url_data = f'https://apidata.mos.ru/v1/datasets/{data_set}/features?api_key={datamos_api}'

### 1.2 Отправляем запрос


Отправляем **GET-запрос** (он используется, когда мы хотим _получить_ данные) и смотрим на **код статуса** — он показывает, как сервер обработал запрос.

Основные статусы:

- **200 OK** — всё прошло успешно, данные получены.
- **400 Bad Request** — ошибка в параметрах запроса.
- **401 / 403** — проблема с доступом (неверный или отсутствующий API-ключ).
- **404 Not Found** — набор данных не найден.
- **500** — ошибка на стороне сервера.


In [ ]:
# Отправляем GET-запрос к API
data_mos = requests.get(url_data)

# Проверяем статус ответа (200 — запрос успешен)
data_mos.status_code

### 1.3 Смотрим на ответ


API вернул ответ в формате **JSON** — это вложенная структура из словарей и списков. Сами объекты лежат в ключе `features`. Посмотрим на ответ целиком:


In [ ]:
# Превращаем ответ в объект Python (словарь) и смотрим на него
data_mos.json()

## 2. Создаём набор пространственных данных


### 2.1 Создаём GeoDataFrame


Объекты из ключа `features` имеют координаты, поэтому мы можем сразу собрать из них **GeoDataFrame**. Метод `from_features` сам прочитает геометрию, а `crs="EPSG:4326"` задаёт систему координат (широта/долгота).


In [ ]:
# Создаём GeoDataFrame напрямую из списка объектов (features)
data_mos_gpd = gpd.GeoDataFrame.from_features(data_mos.json()["features"], crs="EPSG:4326")

# Смотрим на первые строки
data_mos_gpd.head()

### 2.2 Разворачиваем атрибуты в отдельные столбцы


Если посмотреть на таблицу выше, все содержательные признаки объекта (название, адрес и т.д.) спрятаны в одном столбце `attributes` — внутри него лежит **словарь**. Работать с таким столбцом неудобно, поэтому «развернём» каждый словарь в отдельные столбцы и присоединим их к исходной таблице.


In [ ]:
# Разворачиваем словари из столбца 'attributes' в отдельные столбцы.
# .map(str) приводит все значения к строкам, чтобы избежать проблем с разными типами данных
attributes = pd.DataFrame(
    data_mos_gpd['attributes'].values.tolist(),
    index=data_mos_gpd.index
).map(str)

# Присоединяем новые столбцы к исходной таблице и убираем ставший ненужным 'attributes'
data_mos_gpd_final = pd.concat([data_mos_gpd, attributes], axis=1).drop('attributes', axis=1)

# Смотрим на результат
data_mos_gpd_final

### 2.3 Смотрим на результат на карте


In [ ]:
# Быстрая интерактивная карта с полученными объектами
data_mos_gpd_final.explore()

## 3. Сохраняем результат


Сохраним полученные данные в файл, чтобы использовать их дальше (раскомментируйте строку):


In [ ]:
# data_mos_gpd_final.to_file('data_mos_gpd.geojson')

## 4. Итог


На примере **DATA_MOS** мы научились забирать данные с портала открытых данных через API: сформировали запрос, получили ответ в JSON, собрали из него GeoDataFrame и развернули атрибуты в отдельные столбцы.

Такой же подход работает и с другими API — меняются лишь адрес запроса, параметры и структура ответа (её всегда смотрим в документации). **Не забывайте про лимиты** на количество бесплатных запросов.

Успехов!
